# Stock Prediction and Evaluation

In the page on [Cross-Sectional analysis](https://ledatascifi.github.io/ledatascifi-2025/content/05/05d_AssetPricingAnomalyTable1.html), step 2 is to "get your signal, then construct portfolio returns". For the purpose of that page, I just loaded some portfolio returns so that I could focus on show how we run the regression to do the analysis.

On _this page_, I'm going to show the "full stack" asset pricing toolkit:
1. We will build multiple machine learning models.
2. For each model, we will use rolling out-of-sample cross validation to see if the model would have made money trading over time. This has two parts:
   1. First, we train a model on all months up to Dec 31, 2015, and then use it to predict returns throughout 2016. Then we repeat the training with data through Dec 31, 2016 and use the updated model to predict returns throughout 2017, and so on...
   2. Second, to assess whether a model would have made money, we compute the returns they generate if we use them to make a "long-short" portfolio:
      1. We use the predictions as ``signals'' and sort stocks into 5 portfolios (or "bins"), where we think the stocks in bin 1 will do the worst and those in bin 5 the best.
      2. We pretend that we buy the stocks in bin 5 (the "long") and short sell the stocks in bin 1 (the "short"). The short sells finance the buys, so this is a zero-cost portfolio.
      3. This might sound complicated, but in the end, the monthly return of our long-short strategy is just the average return of stocks in bin 5 minus the average return of stocks in bin 1. So you might remember "long-short" as short for "long minus short".
3. For each model, we will subject its performance to a range of assessments.

**What is different from the first version of this notebook.** Everything below is self-contained, and it fixes several look-ahead and bookkeeping problems that made the first version too optimistic or hard to trust:
- **Timing.** The Fama-French factors used as predictors were shifted the wrong way (month $t+1$ factors in month $t$'s row). Now everything measured in month $t$ predicts the return in month $t+1$, and the notebook asserts this, including December to January.
- **Imputation.** Missing signals are filled with the mean of the _same month's_ cross-section (the first version used annual means, which mixed in other months, including future ones).
- **No peeking when preprocessing or tuning.** Dropping nearly-empty variables, winsorizing, scaling and tuning are all learned from training data only, and a fresh pipeline is built for every fit.
- **Tuning respects time.** RidgeCV and LassoCV pick their penalty by training on earlier years and validating on later whole years (never random folds).
- **Portfolios.** Ties in predictions get a documented, symmetric treatment, and the long-short return is exactly bin 5 minus bin 1.
- **Evaluation.** Long-short returns are not reduced by the risk-free rate (they are self-financing), and standard errors are Newey-West (HAC).
- **Saving.** Every fitted model is saved right after it is fit, one file per model and year (the first version saved one file per model type).
- **Data left alone.** Only return months through 2019 are ever loaded. 2020 is an unused buffer year, and 2021 onward is an untouched holdout.

In [ ]:
# Development mode flag - True uses a small random sample of stocks and only the last two test years,
# which is enough to check that the whole notebook runs end to end.
DEV = False

# Standard library imports
import copy
import hashlib
import json
import os
import time
import warnings
from datetime import datetime
from pathlib import Path

# Data processing and visualization
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pandas_datareader as pdr
import seaborn as sns
from tqdm import tqdm

# Statistical and machine learning
import statsmodels.formula.api as smf
import torch
from torch import nn
from sklearn.base import BaseEstimator, RegressorMixin, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Lasso, LinearRegression, Ridge, lasso_path
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Suppress FutureWarnings
warnings.simplefilter(action="ignore", category=FutureWarning)

# ---- Settings -------------------------------------------------------------------------------
FIRST_DATA_YEAR = 1975                                    # first return year we use
FIRST_TEST_YEAR, LAST_TEST_YEAR = (2018, 2019) if DEV else (2000, 2019)
N_VAL_YEARS = 3             # whole years held out (in turn) for tuning inside each training window
MAX_MISSING = 0.9           # drop variables that are missing for more than 90% of the training rows
WINSOR = (0.0025, 0.9975)   # winsorizing quantiles, learned from the training rows
SEED = 0
MLP_SEEDS = 1 if DEV else 5   # the neural net is the average of this many randomly initialised nets
OUT = Path('output_notebook_dev' if DEV else 'output_notebook')
for sub in ['predictions', 'models', 'fits']:
    (OUT / sub).mkdir(parents=True, exist_ok=True)

# Utility function to move columns to the front (left) of a dataframe
def move_column_to_front(df, columns):
    return df[columns + [col for col in df.columns if col not in columns]]

## Assemble data pieces


### Get return data

This is a permno-date dataset with returns from CRSP that I provided to students.

The better method is to programmatically download the data from WRDS via the CRSP API, but this requires a WRDS login, which students do not usually have.

Returns are in **percent**. Only return months from `FIRST_DATA_YEAR` through `LAST_TEST_YEAR` are loaded: later years are never touched, so they cannot influence any modelling choice.

In [ ]:
# This is a permno-date dataset with returns from CRSP. I provided it to students.
# Alternatively, we can download the data from WRDS via the CRSP API.

crsp = pd.read_csv('input_data/crsp_data.csv', usecols=['permno', 'date', 'ret'])
crsp['date'] = pd.to_datetime(crsp['date'])

# If we load more data, we should also load market value and stock price so we can filter
# and do value weighted returns.

# for speedy dev, pick permnos at random (with seed)
if DEV:
    rng = np.random.default_rng(123)
    # to ensure we have enough data, we pick permnos that are in the dataset before 1995 and in the last test year
    # but this makes returns look artificially good because we are picking stocks that survive
    early = crsp.loc[crsp['date'].dt.year < 1995, 'permno'].unique()
    late = crsp.loc[crsp['date'].dt.year == LAST_TEST_YEAR, 'permno'].unique()
    permnos = rng.choice(sorted(set(early) & set(late)), size=300, replace=False)
    crsp = crsp[crsp['permno'].isin(permnos)]

crsp = crsp[crsp['ret'].notnull() & crsp['date'].dt.year.between(FIRST_DATA_YEAR, LAST_TEST_YEAR)].copy()

# To merge with signals (a montly dataset, we will create a datetime column that is the 28th of the month)
# We pick the 28th of the month because it exists in all months
crsp['date'] = pd.to_datetime(crsp['date'].dt.year*100 + crsp['date'].dt.month, format='%Y%m') + pd.DateOffset(days=27)
assert not crsp.duplicated(['permno', 'date']).any()
crsp.info()
crsp

### Some EDA

In [ ]:
window = 12
sns.lineplot(data=(crsp
                  .groupby('date')['ret'].mean().reset_index()
                  .assign(returns_ma=lambda x: x['ret'].rolling(window=window, center=True).mean())
                  ),
             x='date', y='returns_ma').set(title=f'{window} Month Moving Average of Firms Return Over Time', xlabel='Date', ylabel='Average Return')
plt.show()

### Get Fama-French Factors

These are useful as predictors and also for performance evaluation after we construct our portfolios.

**Timing matters here.** The factors have two different jobs, so we keep two versions:
- `ff_factors`: the factor returns in the month they happened. These are only for _evaluating_ portfolios (regressions of portfolio returns on same-month factors).
- `ff_factors_lag`: the _predictor_ version. A factor realised in month $t$ is information we have at the end of month $t$, so it may be used to predict returns in month $t+1$. We therefore move its date **forward** one month, and keep the month it was measured in (`ff_source_date`) so we can check this later. (The first version of this notebook moved the date back by a month, which gave each row the factor of the month it was trying to predict.)

In [ ]:
start = '1964-01-01'
start_date = datetime.strptime(start, '%Y-%m-%d')

ff_file = 'input_data/ff_factors.csv'

if not os.path.exists(ff_file):
    # Get FF5 and momentum
    ff_5 = pdr.get_data_famafrench('F-F_Research_Data_5_Factors_2x3', start=start_date)[0]
    ff_mom = pdr.get_data_famafrench('F-F_Momentum_Factor', start=start_date)[0]
    ff_mom.columns = ['Mom'] # rename

    # combine
    ff_factors = pd.merge(ff_5, ff_mom, left_index=True, right_index=True, how='outer', indicator=True)
    ff_factors = ff_factors.reset_index().rename(columns={"Mkt-RF":"mkt_excess", "Date":"date"})
    ff_factors["date"] = ff_factors["date"].dt.to_timestamp()
    ff_factors.to_csv(ff_file, index=False)
else:
    ff_factors = pd.read_csv(ff_file, parse_dates=["date"])
ff_factors = ff_factors.drop(columns=['_merge'], errors='ignore')

# prep for merging with other datasets
ff_factors["date"] = ff_factors["date"].apply(lambda x: x.replace(day=28)) # set to 28th of month for merging with crsp

# predictor version: measured in month t, used to predict month t+1
FACTORS = ['mkt_excess', 'SMB', 'HML', 'RMW', 'CMA', 'Mom']
ff_factors_lag = ff_factors[['date'] + FACTORS].copy()
ff_factors_lag['ff_source_date'] = ff_factors_lag['date']                    # when the factor was measured
ff_factors_lag['date'] = ff_factors_lag['date'] + pd.DateOffset(months=1)    # the return month it is allowed to predict

ff_factors_lag.query('ff_source_date.dt.month == 12').head(2)   # December factors now sit in January rows

### Download OAP signals zip file from Google Drive

These 200+ signals are stock-level signals that research indicates predict future returns.

The `openassetpricing` package can download all signals easily. But three of the signal variables (Price, Size, and STreversal) require a WRDS login, so we have to input a WRDS login or alternatively use the OAP download function and list all the predictors except those three. This file is [on their site](https://www.openassetpricing.com/data/) as a point and click download ([here](https://drive.google.com/drive/folders/1SSoHGbwgyhRwUCzLE0YWvUlS0DjLCd4k)). Let's use it.

In [ ]:
wide_signal_zipname = 'input_data/signed_predictors_dl_wide.zip'

if not os.path.exists(wide_signal_zipname):
    import gdown   # only needed for this one-time download
    file_id = '1T-nogu88A4hcFXijjftSO41K5P4Hj27y'
    url = f'https://drive.google.com/uc?id={file_id}'
    gdown.download(url, wide_signal_zipname, quiet=False)

### Open Signals from Zip

These signals are measured for each stock in a given month. We want to use them to predict returns in the _following_ month. So we want to merge these signals with the returns for the _following_ month. Example: The January 2010 signals should be merged with the February 2010 returns.

Thus, the most important part of below is that we create a new variable (`date`) that is the month following the signals measurement (`signal_date`). We can use `date` to merge with the `crsp` dataset containing our returns.

The file is big, so we read it in chunks (as 32-bit numbers) and keep only the rows we need. Dropping rows with fewer than 10% of the signals observed only looks at that row itself, so it cannot leak information from other months.

In [ ]:
sig_header = pd.read_csv(wide_signal_zipname, nrows=0).columns
dtypes = {c: np.float32 for c in sig_header if c not in ('permno', 'yyyymm')}
dtypes.update(permno=np.int64, yyyymm=np.int64)

chunks = []
for chunk in pd.read_csv(wide_signal_zipname, dtype=dtypes, chunksize=250_000):
    # signals measured in month t are used for month t+1, so the last one we need is Nov of the last test year
    chunk = chunk[chunk['yyyymm'].between((FIRST_DATA_YEAR - 1)*100 + 12, LAST_TEST_YEAR*100 + 11)]
    if DEV:
        chunk = chunk[chunk['permno'].isin(permnos)]
    # THRESHOLD REQUIREMENT: 10% of signals must be non-missing for a given row of data
    enough = chunk.drop(columns=['permno', 'yyyymm']).notna().mean(axis=1) >= .1
    chunks.append(chunk[enough])
signals = pd.concat(chunks, ignore_index=True)
del chunks

# rename yyyymm to signals so we avoid confusion
signals = signals.rename(columns={'yyyymm': 'yyyymm_signals'})

# signal_date = month the signals were measured; date = the month they are used to predict (one month ahead)
signals['signal_date'] = pd.to_datetime(signals['yyyymm_signals'].astype(str) + '28', format='%Y%m%d')
signals['date'] = signals['signal_date'] + pd.DateOffset(months=1)
signals = move_column_to_front(signals, ['permno', 'date', 'signal_date'])

signals

### Merge all of the Datasets together

After merging, we **assert** the timing: every row's signals and predictor factors were measured exactly one month before the return month. This includes December, whose signals are used for January of the next year.

In [ ]:
print(len(crsp))
crsp = crsp.merge(signals, on=['permno', 'date'], how='inner', validate="one_to_one") # INNER - require the firm to have some signals
print(len(crsp))
crsp = crsp.merge(ff_factors_lag, on='date', how='left', validate="many_to_one")
print(len(crsp))
crsp = crsp.sort_values(['date', 'permno']).reset_index(drop=True)

# ---- timing checks -----------------------------------------------------------------------
def month_index(d):
    d = pd.DatetimeIndex(d)
    return (d.year * 12 + d.month).to_numpy()

assert (month_index(crsp['date']) - month_index(crsp['signal_date']) == 1).all(), 'signals must be measured one month before the return'
assert (month_index(crsp['date']) - month_index(crsp['ff_source_date']) == 1).all(), 'factors must be measured one month before the return'
dec = crsp[crsp['signal_date'].dt.month == 12].iloc[0]
assert dec['date'].month == 1 and dec['date'].year == dec['signal_date'].year + 1
print('December example: signals and factors measured', dec['signal_date'].date(), '-> return month', dec['date'].date())

# value-level check on the factors: the predictor value in a return month equals the factor of the PRIOR month
chk = crsp.drop_duplicates('date').set_index('date')
src = ff_factors.set_index('date').loc[chk['ff_source_date'], FACTORS]
assert np.allclose(chk[FACTORS].to_numpy(), src.to_numpy())

# the holdout policy: nothing after the last test year was ever loaded
assert crsp['date'].dt.year.max() <= LAST_TEST_YEAR
crsp.head()

## Set up Machine Learning (ML) training loop

### Open Documentation to Accurately Map Dtypes

In our machine learning models, we will want to treat continuous signal variables differently than categorical variables. OAP's documentation describes which variables are which kind.

In [ ]:
signal_doc_file = 'input_data/signal_documentation.csv'

if not os.path.exists(signal_doc_file):
    # Download the signal documentation from OpenAP
    import openassetpricing as oap   # only needed for this one-time download
    signal_documentation = oap.OpenAP().dl_signal_doc('pandas')
    signal_documentation.to_csv(signal_doc_file, index=False)
else:
    signal_documentation = pd.read_csv(signal_doc_file)

# need to find a way to add the column type from cat. form to match the acronyms which are the columns of the dataset
signaldoc = signal_documentation[['Acronym', 'Cat.Form']].drop_duplicates()
signaldoc

### ML preliminaries and set up

1. Split the data into X and y
2. Decide which variables are continuous and which are discrete
3. There is **no** global "drop variables that are almost always missing" step any more: which variables are too empty is decided _inside_ each training window (see the preprocessing section), so the choice never uses data from the future.
4. No train/test split by a single date either. Instead, the training loop below trains on every month before year $t$ and predicts year $t$, for each year $t$ from 2000 to 2019. 2020 is a buffer year (not used) and 2021 onward is a holdout that was never loaded.

In [ ]:
# columns to exclude from the training set
exclude = ['ret', 'yyyymm_signals', 'permno', '_merge', 'sinAlgo']

# which variables are continuous and which are discrete?
# this is one way to do it, but note that this won't automatically update if you add your own variables to the dataset
continuous_cols = signaldoc.loc[signaldoc['Cat.Form'] == 'continuous', 'Acronym'].tolist() + FACTORS
discrete_cols = signaldoc.loc[signaldoc['Cat.Form'] == 'discrete', 'Acronym'].tolist()

# filter out columns that are not in the data
continuous_cols = [col for col in continuous_cols if col in crsp.columns and col not in exclude]
discrete_cols = [col for col in discrete_cols if col in crsp.columns and col not in exclude]

# Split the data into X and y. The row index is (permno, date): date = the month whose return we predict
X = crsp[continuous_cols + discrete_cols].copy()
X.index = pd.MultiIndex.from_arrays([crsp['permno'].to_numpy(), crsp['date'].to_numpy()], names=['permno', 'date'])
y = crsp['ret'].to_numpy(dtype=np.float64)              # percent
return_year = crsp['date'].dt.year.to_numpy()
keys = crsp[['permno', 'date', 'signal_date']].copy()   # kept for saving predictions

# Convert to Float32 for better run times (less memory and faster runs)
float_cols = X.select_dtypes('float64').columns
X[float_cols] = X[float_cols].astype(np.float32)

# rows are sorted by date, so each year is one contiguous block: rows [start, end)
bounds = {int(yr): (int(np.searchsorted(return_year, yr, 'left')), int(np.searchsorted(return_year, yr, 'right')))
          for yr in np.unique(return_year)}
print(f'{len(X):,} stock-months, {len(continuous_cols)} continuous and {len(discrete_cols)} discrete candidate variables')

In [ ]:
# del crsp # optional, free up memory

### EDA on the data

In [ ]:
pd.Series(return_year).value_counts().sort_index().plot(kind='line', figsize=(10, 6)).set_title('Number of Obs by Year in the Data')
plt.show()

### More ML preliminaries: the preprocessing steps

All of these steps are **learned from the training rows only** (`fit`) and then applied to the training and prediction rows (`transform`). Nothing here ever looks at the months we are predicting.

1. `MissingnessFilter`: drop variables that are missing for more than 90% of the _training_ rows.
2. `MonthlyCrossSectionalMeanImputer`: fill a missing signal with the average of that signal across all stocks in the _same month_ (cross sectional imputation is a good idea, per Chen et al JFE 2024). All of a row's signals were measured the month before its return, so a same-month average uses only information that was available then. If nobody has the signal in some month, we fall back to the training-sample average. A variable that is missing for _every_ training row has no usable average, so it is dropped.
3. `Winsorizer`: clip each variable at the 0.25th and 99.75th percentile of the training rows.
4. `StandardScaler`, and for the discrete variables a most-frequent imputer followed by one-hot encoding.

Note: the imputer needs `permno` and `date` in the index of `X`, which is why we set them there above.

In [ ]:
def _month_keys(X):
    """Month label of each row, taken from the 'date' index level."""
    d = pd.DatetimeIndex(X.index.get_level_values('date'))
    return (d.year * 12 + d.month).to_numpy()


class MissingnessFilter(BaseEstimator, TransformerMixin):
    """Drop features whose *training* missing share exceeds ``max_missing``."""

    def __init__(self, max_missing=0.9):
        self.max_missing = max_missing

    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        self.missing_share_ = X.isna().mean().to_numpy()
        self.keep_mask_ = self.missing_share_ <= self.max_missing
        return self

    def transform(self, X):
        return pd.DataFrame(X).loc[:, self.keep_mask_]

    def get_feature_names_out(self, input_features=None):
        return self.feature_names_in_[self.keep_mask_]


class MonthlyCrossSectionalMeanImputer(BaseEstimator, TransformerMixin):
    """Fill missing predictors with the mean of the *same month's* cross-section.

    fit() only stores a fallback mean per feature (from the training rows). Features that are
    entirely missing in the training rows have no fallback and are dropped from the output.
    """

    def __init__(self, features_to_impute=None, date_col="date"):
        self.features_to_impute = features_to_impute
        self.date_col = date_col

    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        _month_keys(X)  # checks that the date index level exists
        self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        cols = X.select_dtypes(include=[np.number]).columns if self.features_to_impute is None \
            else [c for c in self.features_to_impute if c in X.columns]
        self.fallback_ = X[list(cols)].mean(skipna=True)           # NaN if all-missing in training
        self.dropped_features_ = list(self.fallback_.index[self.fallback_.isna()])
        self.fallback_ = self.fallback_.dropna()
        self.imputed_features_ = list(self.fallback_.index)
        self.output_features_ = [c for c in X.columns if c not in set(self.dropped_features_)]
        return self

    def transform(self, X):
        X = pd.DataFrame(X)
        month = _month_keys(X)
        out = X[self.output_features_].copy()
        if self.imputed_features_:
            sub = out[self.imputed_features_]
            monthly = sub.groupby(month).transform("mean")         # NaN where a month has no observation
            sub = sub.where(sub.notna(), monthly)
            out[self.imputed_features_] = sub.fillna(self.fallback_[self.imputed_features_])
        return out

    def get_feature_names_out(self, input_features=None):
        return np.asarray(self.output_features_, dtype=object)


class Winsorizer(BaseEstimator, TransformerMixin):
    """Clip each column to training-sample quantile bounds; returns a float64 array."""

    def __init__(self, lower=0.0025, upper=0.9975):
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        A = np.asarray(X)
        self.feature_names_in_ = (np.asarray(X.columns, dtype=object) if hasattr(X, "columns")
                                  else np.arange(A.shape[1]).astype(str).astype(object))
        qs = np.nanquantile(A.astype(np.float64, copy=False), [self.lower, self.upper], axis=0)
        self.lower_bounds_, self.upper_bounds_ = qs[0], qs[1]
        return self

    def transform(self, X):
        return np.clip(np.asarray(X, dtype=np.float64), self.lower_bounds_, self.upper_bounds_)

    def get_feature_names_out(self, input_features=None):
        return np.asarray(input_features if input_features is not None else self.feature_names_in_, dtype=object)

### Set up the preprocessing pipeline

This is a **function**, so every fit gets a brand new, unfitted pipeline. (Re-using one pipeline object across years would be a bug waiting to happen: a fitted object remembers its previous training data.)

In [ ]:
def make_preprocessor():
    """A fresh, unfitted preprocessing pipeline. Call once per fit."""
    numer_pipe = make_pipeline(MissingnessFilter(MAX_MISSING),
                               MonthlyCrossSectionalMeanImputer(),
                               Winsorizer(*WINSOR),
                               StandardScaler())

    cat_pipe = make_pipeline(MissingnessFilter(MAX_MISSING),
                             SimpleImputer(strategy='most_frequent'),
                             OneHotEncoder(handle_unknown='ignore', sparse_output=False))

    # apply the pipelines to the appropriate columns
    return ColumnTransformer(
        transformers=[('numbers', numer_pipe, continuous_cols),
                      ('categoricals', cat_pipe, discrete_cols)],
        remainder='drop')

### Choosing the penalty (alpha) for Ridge and Lasso, using only the past

`Ridge` and `Lasso` shrink coefficients; the penalty `alpha` says how much. The honest way to pick it in a time-series setting:

- Inside a training window (all years before year $t$), hold out each of the last 3 whole years in turn. For each held-out year $v$, fit on the years before $v$ (a fresh preprocessing pipeline each time), and measure the squared prediction error in year $v$.
- Pick the `alpha` with the smallest pooled squared error.
- Then refit with that `alpha` on the whole training window.

Never random folds: a random fold would train on months that come after the months it is tested on.

Trying many alphas naively is slow with millions of rows. The trick: once we have the (centred) cross-product matrices $X'X$ and $X'y$ of the training block, every `alpha` can be evaluated without touching the full data again (Ridge: one eigendecomposition; Lasso: sklearn's coordinate descent path on the Gram matrix). The results equal what `sklearn`'s `Ridge`/`Lasso` give (the package's tests check this).

For Ridge, if the best alpha is at an end of the grid we extend the grid. For Lasso, the grid is a set of fractions of `alpha_max`, the smallest alpha that sets every coefficient to zero.

In [ ]:
def chronological_year_folds(years, n_val_years, min_train_years=5):
    """Expanding-window folds over whole years: [(train_years, val_year), ...] with every train year < val_year."""
    ys = sorted(set(int(v) for v in years))
    folds = [([t for t in ys if t < v], v) for v in ys[-n_val_years:]]
    folds = [(tr, v) for tr, v in folds if len(tr) >= min_train_years]
    assert all(max(tr) < v for tr, v in folds)
    return folds


def sufficient_stats(Z, target, chunk=200_000):
    """Centred X'X, X'y, y'y and means of a (preprocessed) training block, accumulated in float64."""
    n, p = Z.shape
    sx, sxx, sxy, sy, syy = np.zeros(p), np.zeros((p, p)), np.zeros(p), 0.0, 0.0
    for i in range(0, n, chunk):
        xb = np.asarray(Z[i:i + chunk], dtype=np.float64)
        yb = np.asarray(target[i:i + chunk], dtype=np.float64)
        sx += xb.sum(0); sxx += xb.T @ xb; sxy += xb.T @ yb; sy += yb.sum(); syy += float(yb @ yb)
    xm, ym = sx / n, sy / n
    return dict(n=n, p=p, x_mean=xm, y_mean=ym, gram=sxx - n * np.outer(xm, xm),
                xty=sxy - n * xm * ym, yss=syy - n * ym * ym)


def make_fold(Z_tr, y_tr, Z_val, y_val, val_year):
    f = sufficient_stats(Z_tr, y_tr)
    f.update(val_year=int(val_year), Z_val=np.asarray(Z_val, dtype=np.float64), y_val=np.asarray(y_val, dtype=np.float64))
    f['alpha_max'] = float(np.max(np.abs(f['xty'])) / f['n'])   # smallest Lasso alpha that zeroes every coefficient
    return f


def ridge_val_sse(f, alphas):
    """Validation squared error of Ridge(alpha) for every alpha, from the training block's statistics."""
    if 'eig' not in f:
        w, V = np.linalg.eigh(f['gram'])
        f['eig'] = (np.clip(w, 0.0, None), V, V.T @ f['xty'])
    w, V, vty = f['eig']
    B = V @ (vty[:, None] / (w[:, None] + alphas[None, :]))      # coefficients: features x alphas
    b0 = f['y_mean'] - f['x_mean'] @ B                           # intercepts (never penalised)
    pred = f['Z_val'] @ B + b0[None, :]
    return ((f['y_val'][:, None] - pred) ** 2).sum(0)


def lasso_val_sse(f, ratios, tol=1e-4, max_iter=10_000):
    alphas = f['alpha_max'] * ratios
    # sklearn's Gram-based coordinate descent only needs X'X and X'y, so we hand it those (and a stand-in for X)
    stub = np.broadcast_to(np.float64(0.0), (f['n'], f['p']))
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        _, coefs, _ = lasso_path(stub, np.array([np.sqrt(f['yss'])]), alphas=alphas,
                                 precompute=np.ascontiguousarray(f['gram']), Xy=f['xty'],
                                 tol=tol, max_iter=max_iter, check_input=False)
    n_conv = sum(issubclass(w.category, ConvergenceWarning) for w in caught)
    order = np.argsort(-alphas)                    # lasso_path returns alphas in descending order
    B = np.empty_like(coefs); B[:, order] = coefs
    b0 = f['y_mean'] - f['x_mean'] @ B
    pred = f['Z_val'] @ B + b0[None, :]
    return ((f['y_val'][:, None] - pred) ** 2).sum(0), n_conv


def search_ridge(folds, grid=None, max_expansions=3, step=10 ** 0.5):
    grid = np.logspace(-4, 8, 25) if grid is None else np.asarray(grid, float)
    exp = 0
    while True:
        mse = sum(ridge_val_sse(f, grid) for f in folds) / sum(len(f['y_val']) for f in folds)
        b = int(np.argmin(mse))
        if exp >= max_expansions or 0 < b < len(grid) - 1:
            break
        grid = (np.concatenate([grid[0] / step ** np.arange(4, 0, -1), grid]) if b == 0
                else np.concatenate([grid, grid[-1] * step ** np.arange(1, 5)]))   # best alpha at an end: widen
        exp += 1
    return dict(alpha=float(grid[b]), grid=grid.tolist(), val_mse=mse.tolist(), expansions=exp,
                val_years=[f['val_year'] for f in folds])


def search_lasso(folds, alpha_max_outer, n_alphas=30, lo=1e-4, max_expansions=2, tol=1e-4, max_iter=10_000):
    ratios = np.logspace(0, np.log10(lo), n_alphas)
    exp = 0
    while True:
        sse, n_conv = np.zeros(len(ratios)), 0
        for f in folds:
            s, c = lasso_val_sse(f, ratios, tol, max_iter)
            sse += s; n_conv += c
        mse = sse / sum(len(f['y_val']) for f in folds)
        b = int(np.argmin(mse))
        if exp >= max_expansions or b < len(ratios) - 1:
            break
        ratios = np.concatenate([ratios, ratios[-1] * 10 ** (-np.arange(1, 7) / 3.0)])   # best alpha at the small end: extend
        exp += 1
    return dict(alpha=float(ratios[b] * alpha_max_outer), alpha_ratio=float(ratios[b]), val_mse=mse.tolist(),
                expansions=exp, inner_convergence_warnings=int(n_conv), val_years=[f['val_year'] for f in folds])

### Set up any non-default models types you want to try: a neural network

For the neural network we use PyTorch (it runs on a GPU if you have one, and on the CPU otherwise). What we found worked well when we tried a few designs (using only data from before 2005 to choose):

- **Predict the rank, not the return.** Raw returns have a few enormous outliers (a stock that rose 2,000% in a month). The target here is each stock's _rank of return within its month_ (between -0.5 and 0.5). It is robust to outliers, and a month's average return (the market) can't be predicted from it, which is what we want for sorting stocks.
- **Dropout (0.3)** to reduce overfitting. A small net with three hidden layers (256 - 128 - 64).
- **Early stopping on a later period:** the last 3 years of the training window are held out, and we keep the epoch with the best monthly cross-sectional correlation there.
- **Average of several nets** started from different random seeds (stabilises the prediction).

In [ ]:
MLP_SPEC = dict(hidden=(256, 128, 64), dropout=0.3, weight_decay=1e-4, lr=1e-3,
                batch_size=4096, max_epochs=40, patience=5)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def build_net(n_in, spec):
    layers, d = [], n_in
    for h in spec['hidden']:
        layers += [nn.Linear(d, h), nn.ReLU()] + ([nn.Dropout(spec['dropout'])] if spec['dropout'] > 0 else [])
        d = h
    return nn.Sequential(*layers, nn.Linear(d, 1))


def monthly_corr(p, target, month_code, n_months):
    """Mean over months of the cross-sectional Pearson correlation between p and target."""
    p, target = p.double(), target.double()
    s = lambda v: torch.zeros(n_months, device=p.device, dtype=torch.float64).index_add_(0, month_code, v)
    n = s(torch.ones_like(p))
    mp, mt = s(p) / n, s(target) / n
    cov = s(p * target) / n - mp * mt
    vp, vt = s(p * p) / n - mp ** 2, s(target * target) / n - mt ** 2
    c = cov / torch.sqrt((vp * vt).clamp_min(1e-300))
    ok = (n > 2) & (vp > 1e-18) & (vt > 1e-18)
    return float(c[ok].mean())


def predict_net(net, Z, batch=262144):
    net.eval()
    with torch.no_grad():
        return torch.cat([net(Z[i:i + batch]).squeeze(-1) for i in range(0, len(Z), batch)])


def fit_net(Z_tr, t_tr, Z_va, y_va, month_va, n_months_va, spec, seed):
    """Train one net (early stopping on the validation correlation). Returns the best weights."""
    torch.manual_seed(seed)
    net = build_net(Z_tr.shape[1], spec).to(Z_tr.device)
    opt = torch.optim.AdamW(net.parameters(), lr=spec['lr'], weight_decay=spec['weight_decay'])
    gen = torch.Generator(device=Z_tr.device); gen.manual_seed(seed)
    best, best_state, best_epoch = -np.inf, None, -1
    for epoch in range(spec['max_epochs']):
        net.train()
        order = torch.randperm(len(Z_tr), device=Z_tr.device, generator=gen)
        for i in range(0, len(Z_tr), spec['batch_size']):
            idx = order[i:i + spec['batch_size']]
            if len(idx) < 2:
                continue
            opt.zero_grad(set_to_none=True)
            nn.functional.mse_loss(net(Z_tr[idx]).squeeze(-1), t_tr[idx]).backward()
            opt.step()
        score = monthly_corr(predict_net(net, Z_va), y_va, month_va, n_months_va)
        if score > best:
            best, best_epoch = score, epoch
            best_state = {k: v.detach().cpu().clone() for k, v in net.state_dict().items()}
        elif epoch - best_epoch >= spec['patience']:
            break
    return best_state, best_epoch, best


class MLPEnsemble(BaseEstimator, RegressorMixin):
    """Average of several independently trained nets (this object only predicts)."""

    def __init__(self, n_in=0, spec=None, y_scale=1.0, states=None):
        self.n_in, self.spec, self.y_scale, self.states = n_in, spec, y_scale, states

    def __sklearn_is_fitted__(self):
        return self.states is not None

    def fit(self, X=None, y=None):
        raise NotImplementedError('trained by fit_mlp_ensemble(); this object only predicts')

    def predict(self, X):
        Z = torch.as_tensor(np.asarray(X, dtype=np.float32), device=DEVICE)
        total = 0
        for state in self.states:
            net = build_net(self.n_in, self.spec).to(DEVICE)
            net.load_state_dict(state)
            total = total + predict_net(net, Z).double()
        return (total / len(self.states) * self.y_scale).cpu().numpy()


def fit_mlp_ensemble(Z_tr, y_tr, month_tr, Z_va, y_va, month_va, spec, n_seeds, seed):
    """Target = within-month rank of the return (training rows only); validation score uses clipped raw returns."""
    rank = pd.Series(y_tr).groupby(month_tr).rank(pct=True).to_numpy() - 0.5
    scale = float(rank.std())
    lo, hi = np.quantile(y_tr, [0.005, 0.995])                 # clip bounds from the training rows
    t_tr = torch.as_tensor((rank / scale).astype(np.float32), device=DEVICE)
    Zt = torch.as_tensor(Z_tr.astype(np.float32), device=DEVICE)
    Zv = torch.as_tensor(Z_va.astype(np.float32), device=DEVICE)
    yv = torch.as_tensor(np.clip(y_va, lo, hi).astype(np.float32), device=DEVICE)
    mv = torch.as_tensor(pd.factorize(month_va)[0], device=DEVICE)
    nmv = int(mv.max()) + 1
    states, info = [], []
    for k in range(n_seeds):
        st, ep, sc = fit_net(Zt, t_tr, Zv, yv, mv, nmv, spec, seed + k)
        states.append(st); info.append(dict(seed=seed + k, best_epoch=ep, val_corr=sc))
    return MLPEnsemble(n_in=Z_tr.shape[1], spec=spec, y_scale=scale, states=states), info

## Main training loop that generates out-of-sample predictions

For a list of models, train the model on all months before year $t$, then make predictions for year $t$, for $t$ = 2000, ..., 2019.

These out-of-sample predictions are used to build portfolios and evaluate their returns in the next section.

What the loop does each year, in order:
1. Fit a **fresh** preprocessing pipeline on the training rows (years before $t$) only, and transform the training and prediction rows.
2. Tune alpha for RidgeCV and LassoCV with the chronological search above.
3. Fit every model on the training rows. `Ridge` and `Lasso` use the default `alpha=1` as untuned baselines.
4. Predict year $t$, and save the predictions, the fitted pipeline and some metadata **right away** (one file per model and year).

A fit is skipped if its files already exist for the same settings, so you can stop and restart the loop.

In [ ]:
MODEL_NAMES = ['OLS', 'Ridge', 'RidgeCV', 'Lasso', 'LassoCV', 'MLP-rank-256-128-64']   # no underscores: the dashboard parses names

# everything that changes the results goes into one fingerprint; a saved fit is only reused if it matches
CONFIG = dict(first_data_year=FIRST_DATA_YEAR, n_val_years=N_VAL_YEARS, max_missing=MAX_MISSING, winsor=WINSOR,
              ridge_grid_default=np.logspace(-4, 8, 25).tolist(), lasso=(30, 1e-4, 1e-4, 10_000),
              mlp=MLP_SPEC, mlp_seeds=MLP_SEEDS, seed=SEED, dev=DEV, n_rows=len(X),
              n_features=(len(continuous_cols), len(discrete_cols)), last_date=str(keys['date'].max().date()))
CONFIG_HASH = hashlib.sha256(json.dumps(CONFIG, sort_keys=True, default=str).encode()).hexdigest()[:16]
(OUT / 'config.json').write_text(json.dumps({**CONFIG, 'config_hash': CONFIG_HASH}, indent=1, default=str))


def path(kind, model, year, ext):
    return OUT / kind / f'{model}_{year}.{ext}'


def is_done(model, year):
    meta = path('fits', model, year, 'json')
    return (meta.exists() and path('predictions', model, year, 'parquet').exists()
            and path('models', model, year, 'joblib').exists()
            and json.loads(meta.read_text()).get('config_hash') == CONFIG_HASH)   # the .json is written last


def run_year(year):
    todo = [m for m in MODEL_NAMES if not is_done(m, year)]
    if not todo:
        print(f'[{year}] already complete, skipping')
        return
    a, b = bounds[year]                                   # test rows are [a, b); training rows are [0, a)
    assert (return_year[:a] < year).all() and (return_year[a:b] == year).all()
    assert keys['date'].iloc[a - 1] < keys['date'].iloc[a]
    assert keys['signal_date'].iloc[a - 1] < keys['date'].iloc[a]
    X_tr, X_te, y_tr, y_te = X.iloc[:a], X.iloc[a:b], y[:a], y[a:b]

    # 1. a fresh preprocessing pipeline, fit on the training rows only
    t0 = time.time()
    pre = make_preprocessor().fit(X_tr)
    Z_tr, Z_te = pre.transform(X_tr), pre.transform(X_te)
    assert np.isfinite(Z_tr).all() and np.isfinite(Z_te).all()

    # 2. tune alpha chronologically (whole validation years, preprocessing refit on each inner training block)
    searches = {}
    if 'RidgeCV' in todo or 'LassoCV' in todo:
        folds = []
        for _, v in chronological_year_folds([yr for yr in bounds if yr < year], N_VAL_YEARS):
            s_v, e_v = bounds[v]
            pre_f = make_preprocessor().fit(X.iloc[:s_v])
            folds.append(make_fold(pre_f.transform(X.iloc[:s_v]), y[:s_v], pre_f.transform(X.iloc[s_v:e_v]), y[s_v:e_v], v))
        if 'RidgeCV' in todo:
            searches['RidgeCV'] = search_ridge(folds)
        if 'LassoCV' in todo:
            alpha_max = float(np.max(np.abs(sufficient_stats(Z_tr, y_tr)['xty'])) / len(y_tr))
            searches['LassoCV'] = search_lasso(folds, alpha_max)
        del folds
    t_shared = time.time() - t0

    # 3. fit every model that still needs fitting, 4. predict and save immediately
    for m in todo:
        t1 = time.time()
        extra = {}
        if m == 'OLS':
            est = LinearRegression().fit(Z_tr, y_tr)
        elif m == 'Ridge':
            est = Ridge(alpha=1.0).fit(Z_tr, y_tr)
        elif m == 'RidgeCV':
            est = Ridge(alpha=searches[m]['alpha']).fit(Z_tr, y_tr)
        elif m in ('Lasso', 'LassoCV'):
            alpha = 1.0 if m == 'Lasso' else searches[m]['alpha']
            with warnings.catch_warnings(record=True) as wl:
                warnings.simplefilter('always')
                est = Lasso(alpha=alpha, precompute=True, max_iter=10_000, tol=1e-4).fit(Z_tr, y_tr)
            extra['convergence_warnings'] = sum(issubclass(w.category, ConvergenceWarning) for w in wl)
        elif m.startswith('MLP'):
            # the last N_VAL_YEARS years before `year` are the early-stopping set; the nets train on the earlier years
            s_va = bounds[year - N_VAL_YEARS][0]
            dates = keys['date']
            est, info = fit_mlp_ensemble(Z_tr[:s_va], y_tr[:s_va], month_index(dates.iloc[:s_va]),
                                         Z_tr[s_va:], y_tr[s_va:], month_index(dates.iloc[s_va:a]),
                                         MLP_SPEC, MLP_SEEDS, SEED + 1000 * year)
            extra['nets'] = info
        else:
            raise ValueError(m)
        pred = est.predict(Z_te)
        assert np.isfinite(pred).all()

        pipe = Pipeline([('preprocess', copy.deepcopy(pre)), ('model', est)])    # this fit's own copies
        first_month = (keys['date'].iloc[a:b] == keys['date'].iloc[a]).to_numpy().sum()   # a whole month: the imputer needs it
        assert np.allclose(pipe.predict(X.iloc[a:a + first_month]), pred[:first_month], rtol=1e-3, atol=1e-4), 'saved pipeline != predictions'
        joblib.dump(pipe, path('models', m, year, 'joblib'), compress=3)
        pd.DataFrame({'permno': keys['permno'].to_numpy()[a:b], 'date': keys['date'].to_numpy()[a:b],
                      'ret': y_te, 'pred': pred}).to_parquet(path('predictions', m, year, 'parquet'))
        meta = dict(model=m, year=year, config_hash=CONFIG_HASH, train_rows=int(a), test_rows=int(b - a),
                    train_end=str(keys['date'].iloc[a - 1].date()), test_start=str(keys['date'].iloc[a].date()),
                    n_features=int(Z_tr.shape[1]), alpha=getattr(est, 'alpha', None), pred_std=float(np.std(pred)),
                    seconds=dict(shared=round(t_shared, 1), fit=round(time.time() - t1, 1)),
                    search=searches.get(m), **extra)
        path('fits', m, year, 'json').write_text(json.dumps(meta, indent=1, default=float))   # written last = "done"
    del Z_tr, Z_te


for year in range(FIRST_TEST_YEAR, LAST_TEST_YEAR + 1):
    print(f'=== test year {year}: train on {FIRST_DATA_YEAR}-{year - 1} ===', flush=True)
    run_year(year)

Load all the saved predictions into one table (one `pred_<model>` column per model) and check it:

In [ ]:
years = range(FIRST_TEST_YEAR, LAST_TEST_YEAR + 1)
prediction_output = None
for m in MODEL_NAMES:
    d = pd.concat([pd.read_parquet(path('predictions', m, yr, 'parquet')) for yr in years], ignore_index=True)
    d = d.rename(columns={'pred': f'pred_{m}'})
    if prediction_output is None:
        prediction_output = d
    else:   # every model must have predicted exactly the same stock-months
        assert (d[['permno', 'date']].to_numpy() == prediction_output[['permno', 'date']].to_numpy()).all()
        assert np.array_equal(d['ret'].to_numpy(), prediction_output['ret'].to_numpy())
        prediction_output[f'pred_{m}'] = d[f'pred_{m}'].to_numpy()
prediction_output = prediction_output.sort_values(['date', 'permno']).reset_index(drop=True)

assert not prediction_output.duplicated(['permno', 'date']).any()
assert np.isfinite(prediction_output.drop(columns=['permno', 'date']).to_numpy()).all()
prediction_output.to_csv(OUT / 'prediction_output.csv', index=False)

Some speedy EDA of the predictions:

In [ ]:
prediction_output.groupby(prediction_output['date'].dt.year)['permno'].nunique().plot(kind='line').set_title('Number of Firms by Year in the OOS Predictions')
display(prediction_output)
print('\n\n')
display(prediction_output.describe())

## Make portfolios and determine portfolio returns

For a given model (column in the prediction df which are your new signals), compute the portfolio returns by sorting the predictions into quintiles and computing the average return of each quintile.

**What to do with ties.** If several stocks have the same prediction (this happens, e.g., when a Lasso sets every coefficient to zero and predicts the same number for everyone), a simple sort has to break the tie somehow, and the usual shortcuts (`rank(method='min')`) can leave a bin empty. We use a fair rule that never looks at returns: stocks with identical predictions share their positions in the ranking equally, so each tied stock puts the same fractional weight in each bin it straddles. Every bin always holds exactly one fifth of the stocks' weight. If _all_ predictions are identical in a month, there is no signal: every bin earns the average stock's return, and the long-short return is 0 for that month. We keep those months (as zeros) and count them.

In [ ]:
# convert returns to decimals
prediction_output['ret'] = prediction_output['ret'] / 100
assert prediction_output['ret'].min() >= -1, 'returns below -100%: unit problem'
model_names = [c[5:] for c in prediction_output.columns if c.startswith('pred_')]
model_names

In [ ]:
TIE_DECIMALS = 12     # predictions that differ by less than 1e-12 are float noise


def bin_weights(df, pred_col, nport=5):
    """W[i, k] = how much of stock i sits in bin k+1; each (month, bin) column sums to n_stocks / nport."""
    p = df[pred_col].to_numpy(dtype=np.float64)
    if not np.isfinite(p).all():
        raise ValueError(f'non-finite predictions in {pred_col}')
    g = pd.DataFrame({'d': df['date'].to_numpy(), 'p': np.round(p, TIE_DECIMALS)}).groupby('d', sort=False)['p']
    first, last = g.rank(method='min').to_numpy() - 1.0, g.rank(method='max').to_numpy()   # positions a tie group occupies
    n = g.transform('size').to_numpy().astype(np.float64)
    W = np.empty((len(df), nport))
    for k in range(nport):
        lo, hi = k * n / nport, (k + 1) * n / nport                        # the part of the ranking owned by bin k+1
        W[:, k] = np.clip(np.minimum(last, hi) - np.maximum(first, lo), 0, None) / (last - first)
    return W, pd.DataFrame({'d': df['date'].to_numpy(), 'tie_size': last - first, 'p': np.round(p, TIE_DECIMALS)})


def portfolio_returns(df, W, nport=5):
    """Equal-weighted bin returns (decimals) + stats per bin and month + normalised weights (each sums to 1)."""
    r, d = df['ret'].to_numpy(dtype=np.float64), df['date'].to_numpy()
    n = pd.Series(d).groupby(d).transform('size').to_numpy().astype(np.float64)
    wn = W / (n / nport)[:, None]
    means, tall = [], []
    for k in range(nport):
        t = pd.DataFrame({'d': d, 'w': wn[:, k], 'wr': wn[:, k] * r, 'wr2': wn[:, k] * r * r, 'in_bin': wn[:, k] > 1e-12})
        g = t.groupby('d')
        mean = g['wr'].sum()
        std = np.sqrt((g['wr2'].sum() - mean ** 2).clip(lower=0))
        tall.append(pd.DataFrame({'bin': k + 1, 'ret': mean, 'std': std, 'nfirms': g['in_bin'].sum(), 'weight_sum': g['w'].sum()}).rename_axis('date').reset_index())
        means.append(mean.rename(k + 1))
    wide = pd.concat(means, axis=1).rename_axis('date')
    wide['LS'] = wide[nport] - wide[1]                      # long-short = bin 5 minus bin 1, by construction
    return wide, pd.concat(tall, ignore_index=True), wn


def weight_turnover(df, wn, nport=5):
    """% buy: weight of this month's bin not held last month; % sell: last month's weight no longer held; % hold = 1 - % buy."""
    base = pd.DataFrame({'permno': df['permno'].to_numpy(), 'date': df['date'].to_numpy()})
    rows = []
    for k in range(nport):
        piv = base.assign(w=wn[:, k]).pivot(index='date', columns='permno', values='w').fillna(0.0)
        diff = piv.diff().iloc[1:]
        buy, sell = diff.clip(lower=0).sum(1), (-diff).clip(lower=0).sum(1)
        rows.append(pd.DataFrame({'bin': k + 1, '% buy': buy, '% sell': sell, '% hold': 1 - buy}).reset_index())
    return pd.concat(rows, ignore_index=True)

In [ ]:
returns, port_stats, turnover, tie_rows = [], [], [], []
for model in model_names:
    W, tie_key = bin_weights(prediction_output, f'pred_{model}', nport=5)
    wide, tall, wn = portfolio_returns(prediction_output, W, nport=5)

    # checks: LS is bin 5 - bin 1, every bin always has stocks, every bin's weights sum to one
    assert np.allclose(wide['LS'], wide[5] - wide[1], rtol=0, atol=1e-15)
    assert (tall['nfirms'] >= 1).all()
    assert np.allclose(tall['weight_sum'], 1.0)

    wide.columns = [f'ret_{model}_bin{c}' for c in wide.columns]
    returns.append(wide)
    tall['model'] = model
    port_stats.append(move_column_to_front(tall.drop(columns='weight_sum'), ['model', 'bin', 'date']))
    t = weight_turnover(prediction_output, wn)
    t['model'] = model
    turnover.append(t)

    by_month = tie_key.groupby('d')
    tie_rows.append(dict(model=model, share_of_stocks_in_ties=(tie_key['tie_size'] > 1).mean(),
                         months_with_no_dispersion=int((by_month['p'].nunique() == 1).sum()), months=by_month.ngroups))

returns_wide_df = pd.concat(returns, axis=1)
port_stats_tall_df = pd.concat(port_stats, ignore_index=True)
turnover_df = move_column_to_front(pd.concat(turnover, ignore_index=True), ['model', 'bin', 'date'])
turnover_df = turnover_df[['model', 'bin', 'date', '% buy', '% sell', '% hold']].sort_values(['model', 'bin', 'date'])
pd.DataFrame(tie_rows)    # ties and months with no signal at all, by model

In [ ]:
returns_wide_df.to_csv(OUT / 'returns_wide.csv', index=True)
port_stats_tall_df.to_csv(OUT / 'port_stats_tall.csv', index=False)
turnover_df.to_csv(OUT / 'turnover.csv', index=False)

In [ ]:
X.reset_index().groupby('date')['permno'].nunique().plot(kind='line').set_title('Number of Firms by Year')
plt.show()

In [ ]:
sns.lineplot(data=port_stats_tall_df.query('model=="RidgeCV"'), hue='bin', x='date', y='nfirms'
            ).set(title='Number of Firms in Each Portfolio Over Time', xlabel='Date', ylabel='Number of Firms')
plt.show()

## Portfolio Analysis

### "Table 1" Results, for each portfolio

Alphas come from regressions of portfolio returns on factors measured **in the same month** (`ff_factors`, not the lagged predictor version), in percent per month. Details:
- For the long-only bins, the dependent variable is the return minus the risk-free rate (an excess return). The long-short portfolio is self-financing, so we do **not** subtract the risk-free rate from it.
- The t-statistics use Newey-West (HAC) standard errors with $\lfloor 4(T/100)^{2/9} \rfloor$ lags (4 for the 240 months here), because monthly portfolio returns can be autocorrelated.

In [ ]:
def table_1(df, model_tag):
    portfolios = [c for c in df.columns if c.startswith(f'ret_{model_tag}_bin')]    # exactly this model's bins
    reg_df = (df[portfolios] * 100).merge(ff_factors.set_index('date'), left_index=True, right_index=True, how='left')   # percent
    lags = int(np.floor(4 * (len(reg_df) / 100) ** (2 / 9)))                        # Newey-West lag rule of thumb

    # define factor model formulas (these are the right side of the regrssion formulas)
    # these are how formulas are specified for statsmodel's formula api
    factor_models = {
        'r^e':  '1',
        'CAPM': 'mkt_excess',
        'FF3':  'mkt_excess + SMB + HML',
        'FF4':  'mkt_excess + SMB + HML + Mom',
        'FF5':  'mkt_excess + SMB + HML + RMW + CMA',
        'FF6':  'mkt_excess + SMB + HML + RMW + CMA + Mom'
    }

    # pre built output table
    index = pd.MultiIndex.from_product([factor_models.keys(), ['alpha', 't-stat']], names=['Model', 'Metric'])
    results = pd.DataFrame(index=index, columns=portfolios, dtype=float)

    # Run regressions for each portfolio and model
    for portfolio in portfolios:
        reg_df['y'] = reg_df[portfolio] - (0 if portfolio.endswith('binLS') else reg_df['RF'])   # excess return, except long-short
        for model_name, formula in factor_models.items():
            reg = smf.ols(formula=f'y ~ {formula}', data=reg_df).fit(cov_type='HAC', cov_kwds={'maxlags': lags})
            results.at[(model_name, 'alpha'), portfolio] = reg.params['Intercept']
            results.at[(model_name, 't-stat'), portfolio] = reg.tvalues['Intercept']

    return results

In [ ]:
table_1s = {}
for model in model_names:
    print(f'\n\nModel: {model}')
    table_1s[model] = table_1(returns_wide_df, model)
    display(table_1s[model])
    print('\n\n')

### Returns plot

In [ ]:
plot_df = returns_wide_df.filter(like='_binLS')*100
plot_df.columns = [c.replace('ret_', '').replace('_binLS','') for c in plot_df.columns]
sns.lineplot(plot_df
             ).set(title=f'L-S Portfolio Returns',
                   xlabel='',
                   ylabel=f'Monthly Return As Percent')
plt.show()

In [ ]:
window = 12
plot_df = returns_wide_df.filter(like='_binLS')*100
plot_df.columns = [c.replace('ret_', '').replace('_binLS','') for c in plot_df.columns]
sns.lineplot(plot_df.rolling(window=window, center=True).mean()
             ).set(title=f'L-S Portfolio Returns',
                   xlabel='',
                   ylabel=f'{window} Month Moving Average Return As Percent')
plt.show()

### Cumulative returns plot

Wealth is `(1 + r).cumprod()` of the long-short return, starting from \$1. This is a long-short book that is long \$1 and short \$1 for each \$1 of equity, rebalanced monthly, **before** transaction costs, borrowing fees and financing effects.

In [ ]:
plot_df = (1+returns_wide_df.filter(like='_binLS')).cumprod()
# add a month before the first date and set values to one for the first month
row1 = pd.DataFrame(index=[plot_df.index[0] - pd.DateOffset(months=1)], data={c:[1] for c in plot_df.columns})
plot_df = pd.concat([row1, plot_df])

# now plot
plot_df.columns = [c.replace('ret_', '').replace('_binLS','') for c in plot_df.columns]
sns.lineplot(plot_df).set(title=f'L-S Portfolio Cumulative Returns',
                   xlabel='',
                   ylabel=f'Indexed to 1 at {plot_df.index[1].strftime("%Y-%m-01")}')
plt.yscale('log')
plt.show()

In [ ]:
# out of sample performance of the long-short portfolios (no risk-free rate to remove: they are self-financing)
def performance(r):
    wealth = (1 + r).cumprod()
    w = np.concatenate([[1.0], wealth.to_numpy()])
    return pd.Series({'CAGR': wealth.iloc[-1] ** (12 / len(r)) - 1, 'ann. mean': 12 * r.mean(),
                      'ann. vol': np.sqrt(12) * r.std(), 'Sharpe': np.sqrt(12) * r.mean() / r.std(),
                      'max drawdown': (w / np.maximum.accumulate(w) - 1).min(), 'ending wealth': wealth.iloc[-1]})

ls = returns_wide_df.filter(like='_binLS')
ls.columns = [c.replace('ret_', '').replace('_binLS', '') for c in ls.columns]
perf = pd.concat({start: ls[ls.index >= start].apply(performance).T for start in ['2000-01-01', '2005-01-01', '2010-01-01']},
                 names=['evaluation starts'])
perf.round(3)

## Save turnover metrics by portfolio

For each portfolio and month: what share of the portfolio (by weight) is newly bought, what share of last month's holdings is sold, and what share is held. Computed from the bin weights above, so it is consistent with the returns.

In [ ]:
turnover_df.to_csv(OUT / 'turnover.csv', index=False)
turnover_df

In [ ]:
# present decimals as % via styler
turnover_df.set_index(['model','bin']).filter(like='%').groupby(level=['model','bin']).mean().round(2).style.format(
    {'% buy': '{:.2%}', '% sell': '{:.2%}', '% hold': '{:.2%}'},
    na_rep='-'
).set_caption('Average Turnover by Portfolio and Model')

## Model Output - the saved model objects

There is nothing extra to run here any more: every fitted pipeline (preprocessing + model) was saved right after it was fit, as `models/<model>_<year>.joblib`. You get one file per model and test year, each trained only on the years before it. The dashboard lets you pick the year and shows the coefficients or, for the neural nets, a rough variable importance.

To reload one and predict (the class definitions above need to be in scope, because the pickles refer to them):

In [ ]:
pipe = joblib.load(path('models', 'RidgeCV', LAST_TEST_YEAR, 'joblib'))
pipe.predict(X.iloc[bounds[LAST_TEST_YEAR][0]:bounds[LAST_TEST_YEAR][0] + 1000])[:5]